# 1 · MiniPlace: peers organize themselves

**North Mini Code peers · one shared 384×288 canvas**

The agents reconstruct a Canadian flag above the Cohere logo. Each peer starts with the same goal and chooses its own work, collaborators, and boundaries. Work announcements are advisory; the models decide how to cooperate.

The dashboard shows their actual actions. Completion requires every pixel to match the reference.

## 1. Setup

From the project folder, run `uv sync --locked` and `uv run jupyter lab`. Use the Python 3 kernel and run the cells in order. Restart the kernel after updating the helper files.

For Colab, upload the three `miniplace_*.py` helpers and the `assets/` folder alongside this notebook. The install cell below installs the required libraries.

Provide a [Cohere API key](https://dashboard.cohere.com/api-keys) with access to `north-mini-code-1-0`, using `COHERE_API_KEY`, `CO_API_KEY`, or the hidden prompt. All model inputs are text-only.

In [ ]:
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
try:
    sdk_ready = version('cohere') == '7.1.1' and all(version(package) for package in ('anywidget', 'jsonschema'))
except PackageNotFoundError:
    sdk_ready = False
if not sdk_ready:
    uv_binary = shutil.which('uv')
    if uv_binary:
        command = [uv_binary]
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        command = [sys.executable, '-m', 'uv']
    subprocess.run(command + ['pip', 'install', '--python', sys.executable, 'cohere==7.1.1', 'anywidget>=0.9,<1', 'jsonschema>=4.23,<5'], check=True)

In [ ]:
import os
from getpass import getpass
import cohere
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
from miniplace_runtime import Config, Dashboard, Studio, load_cohere_mural

api_key = os.getenv('COHERE_API_KEY') or os.getenv('CO_API_KEY') or getpass('Cohere API key: ')
if not api_key.strip():
    raise ValueError('Enter a Cohere API key.')
client = cohere.AsyncClientV2(api_key=api_key.strip(), timeout=60, max_retries=0)
del api_key

## 2. Shared goal, budgets, and guidance

The local reference has **110,592 pixels and 14 colors**. The blank canvas starts with **60,432 incorrect pixels**. Every agent receives the same reference information.

Adjust the team size, request limits, and wall-time budget below. Each agent retains its full conversation in memory; active requests use recent complete exchanges and structured memory after the context threshold. `recall_context` retrieves earlier coordination.

In [ ]:
config = Config(
    async_paint=True, raw_paint_tools=False, thinking_budget=512,
    context_soft_limit=100_000, context_recent_turns=4,
    painters=16, worker_concurrency=16,
    max_agent_calls=500, max_seconds=180 * 8,
    pixel_delay=0.001, ui_fps=16,
)
target, palette = load_cohere_mural()
PEER_GUIDANCE = (
    'Use the LATEST uncovered_errors and work_advice, not old reference previews or remembered completed scopes. '
    'Choose a useful bounded scope yourself from current exact native [start,stop) uncovered error spans. Announce it once. '
    'An already_correct rejection creates NO work item: move to a different current error band rather than reporting progress or repeating those coordinates. '
    'If work_advice says paint_work, immediately paint_work(own_work_id); later proposals should yield to your earlier accepted plan. '
    'If resolve_overlap, prefer a different uncovered scope instead of holding a meeting. '
    'If work_in_flight, do not queue the item again. If finish_scope, report_done and choose more useful work while the shared target remains unfinished. '
    'Paint_work executes the entire chosen work item asynchronously; no tiny strip actions or repeated coordinates are needed. '
    'Keep public messages short and factual, and use direct negotiation only for a concrete unavoidable handoff. '
    'Do not revisit old correct regions for reassurance. Every decision should move useful work forward or resolve a specific blocker.'
)
preview_studio = Studio('peer', config=config, target=target, palette=palette)
preview = Dashboard(preview_studio)  # Reference preview; no model calls.

## 3. Tools the peers choose

| Tool | Model decision |
|---|---|
| `announce_work` | Choose a bounded scope; receive its `work_id`, real progress, and overlap feedback |
| `paint_work` | Paint that exact work ID in the background, skipping correct pixels |
| `send_message` | Choose a recipient, message, and urgency |
| `post_update` | Publish a shared milestone, blocker, or handoff |
| `release_plan` | Withdraw an intention |
| `watch_canvas` | Pause briefly and observe other work |
| `report_done` | Close a scope and continue useful work while errors remain |
| `finish_mural` | Request exact verification of the entire canvas |

A work item is bounded by `max_plan_pixels` (16,384 by default). The reference brush executes the region chosen by the model. Plans do not restrict other agents' access.

For a raw-painting experiment, set `raw_paint_tools=True` to expose standalone inspection and custom pixel tools, or `reference_brush=False` to require pixel transcription.

In [ ]:
[tool['function']['name'] for tool in preview_studio.tools_for('P01')]

## 4. Run the demo

Peers start together. Each has a bounded brush queue, allowing inference and communication to continue during painting. Urgent messages prompt reconsideration; ordinary messages arrive at the next decision.

Use **Full screen**, **Work board**, and **Expand graph** to follow coordination. Select an agent to see its task, request phase, last-output age, and deadline. Scrolling up pauses log following; **Resume live** returns to new events.

Each API request has a 60-second total deadline by default (`request_timeout`), with bounded retries. Only complete, validated tool calls execute. The run stops on exact completion or its configured limits.

In [ ]:
peer_studio = Studio('peer', client=client, config=config, target=target, palette=palette)
peer_studio.extra_prompts['peer'] = PEER_GUIDANCE
peer_result = await peer_studio.run()

In [ ]:
peer_studio.summary()
check = peer_studio.check()
{key: check[key] for key in ('valid', 'matched', 'total', 'wrong')}

## 5. Explore the result

- Inspect `peer_studio.messages`, `peer_studio.consumed`, and `peer_studio.plan_history` to compare intentions with actions.
- Replay recent frames with `await peer_studio.view.replay()`; this makes no model calls.
- Edit `PEER_GUIDANCE` and start a fresh run to compare coordination strategies.

Outcomes vary. Use the exact pixel check to distinguish completion from partial progress.